# Lab 01: Spark Fundamentals

## Objectives
- Understand Spark architecture and components (Driver, Executor, Cluster Manager)
- Create and manipulate Spark DataFrames
- Perform basic data transformations and actions
- Understand the difference between RDDs, DataFrames, and Datasets
- Use Spark SQL for data analysis
- Navigate the Spark UI to inspect jobs, stages, and tasks

## Domain Coverage
- **Spark Architecture and Components — 15%**
- **DataFrame API — 30%**

## Estimates
- **Time:** ~60-90 minutes
- **Difficulty:** Beginner
- **Prerequisites:** Docker, Python 3.8+

![Spark Architecture](../assets/diagrams/lab-01-spark-architecture.mmd)

In [ ]:
# Import required libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, upper, lower
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

In [ ]:
# Create Spark session
spark = SparkSession.builder \
    .appName("SparkFundamentals") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "1g") \
    .config("spark.executor.cores", "1") \
    .getOrCreate()

# Verify Spark session
print(f"Spark version: {spark.version}")
print(f"Spark master: {spark.conf.get('spark.master')}")
print(f"App name: {spark.conf.get('spark.app.name')}")

## A. SparkSession — Your Entry Point

Every Spark application starts with a `SparkSession`. It provides access to the DataFrame API, SQL engine, and cluster configuration.

In [ ]:
# Inspect Spark configuration
print(f"Default parallelism: {spark.conf.get('spark.default.parallelism')}")
print(f"Shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"AQE enabled: {spark.conf.get('spark.sql.adaptive.enabled')}")

## B. Creating Your First DataFrame

DataFrames are distributed collections of data organized into named columns. They are the primary abstraction in Spark.

In [ ]:
# Create sample data
data = [
    ("Alice", 25, "Engineering", 75000),
    ("Bob", 30, "Marketing", 65000),
    ("Charlie", 35, "Engineering", 85000),
    ("Diana", 28, "Sales", 60000),
    ("Eve", 32, "Marketing", 70000)
]

# Define schema
schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("department", StringType(), True),
    StructField("salary", DoubleType(), True)
])

# Create DataFrame
df = spark.createDataFrame(data, schema)

# Display the DataFrame
df.show()

In [ ]:
# Explore DataFrame schema and structure
df.printSchema()
print(f"\nColumn names: {df.columns}")
print(f"\nRow count: {df.count()}")

## C. Transformations vs Actions

Spark operations are divided into two categories:
- **Transformations**: Lazy operations that build up the computation plan
- **Actions**: Eager operations that trigger actual computation

In [ ]:
# Transformations (lazy - no computation happens yet)
filtered_df = df.filter(col("age") > 30)
selected_df = df.select("name", "department")
transformed_df = df.withColumn("salary_category", when(col("salary") > 70000, "High").otherwise("Low"))

print("Transformations defined (lazy evaluation)")

In [ ]:
# Actions (eager - trigger computation)
print("Count action:")
print(df.count())

print("\nShow action:")
df.show()

print("\nCollect action:")
result = df.collect()
print(f"Collected {len(result)} rows")

## D. Basic DataFrame Operations

Let's practice common DataFrame operations.

In [ ]:
# Filter operations
print("Employees older than 30:")
df.filter(col("age") > 30).show()

print("\nEngineering department:")
df.filter(col("department") == "Engineering").show()

In [ ]:
# Select operations
print("Select specific columns:")
df.select("name", "salary").show()

print("\nSelect with expressions:")
df.select(col("name"), col("salary") * 1.1).show()

In [ ]:
# Add/modify columns
df_with_bonus = df.withColumn("bonus", col("salary") * 0.10)
df_with_bonus.show()

# Conditional column
df_with_category = df.withColumn(
    "age_group",
    when(col("age") < 30, "Young").when(col("age") < 35, "Middle").otherwise("Senior")
)
df_with_category.show()

## E. Spark SQL Operations

You can also use SQL to query DataFrames by registering them as temporary views.

In [ ]:
# Register DataFrame as a temporary view
df.createOrReplaceTempView("employees")

# Run SQL queries
print("All employees:")
spark.sql("SELECT * FROM employees").show()

print("\nAverage salary by department:")
spark.sql("""
    SELECT department, AVG(salary) as avg_salary
    FROM employees
    GROUP BY department
    ORDER BY avg_salary DESC
""").show()

## F. Reading and Writing Data

Spark supports multiple data formats for reading and writing.

In [ ]:
# Write DataFrame to different formats
df.write.mode("overwrite").csv("data/csv/employees.csv")
df.write.mode("overwrite").json("data/json/employees.json")
df.write.mode("overwrite").parquet("data/parquet/employees.parquet")

print("Data written to CSV, JSON, and Parquet formats")

In [ ]:
# Read data back
df_csv = spark.read.csv("data/csv/employees.csv", header=True, inferSchema=True)
df_json = spark.read.json("data/json/employees.json")
df_parquet = spark.read.parquet("data/parquet/employees.parquet")

print("Read from CSV:")
df_csv.show()

print("\nRead from Parquet:")
df_parquet.show()

## G. Navigating the Spark UI

Open the **Spark UI** at http://localhost:4040 to inspect:

| Tab | What to Look For |
|-----|-----------------|
| **Jobs** | Number of jobs triggered, their status, and duration |
| **Stages** | Shuffle read/write bytes, number of tasks per stage |
| **Storage** | Cached DataFrames |
| **Environment** | All Spark configuration properties |
| **Executors** | Memory usage, active tasks, shuffle metrics |
| **SQL/DataFrame** | Visual query plans for DataFrame operations |

In [ ]:
# Trigger a job and inspect it in Spark UI
complex_query = (
    df.filter(col("age") > 25)
    .filter(col("salary") > 65000)
    .groupBy("department")
    .agg(
        count("*").alias("employee_count"),
        avg("salary").alias("avg_salary")
    )
    .orderBy(col("avg_salary").desc())
)

# Explain the query plan
print("=== Physical Plan ===")
complex_query.explain()

# Execute the query (triggers job)
complex_query.show()

## Exam-Style Review

**Q1.** What is the role of the SparkSession in a Spark application?
- A) It stores data partitions
- B) It is the entry point for programming Spark with DataFrames and SQL
- C) It manages cluster resources and assigns workers
- D) It only executes tasks assigned by the cluster manager

**Q2.** What is the difference between transformations and actions?
- A) Transformations are eager, actions are lazy
- B) Transformations are lazy, actions are eager
- C) They are identical
- D) Transformations modify data, actions read data

**Q3.** Which of the following is an action operation?
- A) filter()
- B) select()
- C) count()
- D) withColumn()

**Q4.** How do you execute SQL queries on a DataFrame?
- A) Use the df.sql() method
- B) Register the DataFrame as a temporary view and use spark.sql()
- C) Convert DataFrame to SQL string
- D) SQL cannot be used with DataFrames

### Answers
- **Q1: B** — SparkSession is the entry point for programming Spark with DataFrames and SQL.
- **Q2: B** — Transformations are lazy (build computation plan), actions are eager (trigger computation).
- **Q3: C** — count() is an action. filter(), select(), and withColumn() are transformations.
- **Q4: B** — Register DataFrame as a temporary view using createOrReplaceTempView(), then use spark.sql().

## Key Takeaways
- SparkSession is the entry point for all Spark operations
- DataFrames are distributed collections of data organized into named columns
- Transformations are lazy operations that build up the computation plan
- Actions trigger actual computation and return results
- Spark SQL can be used by registering DataFrames as temporary views
- The Spark UI provides insights into job execution and performance

**Next:** [Lab 02 — Data Loading & Transformation](chapter-02-data-loading-transformation.ipynb)

In [ ]:
# Cleanup
spark.catalog.dropTempView("employees")
print("Lab 01 complete. Temporary views cleaned up.")